# Week 11 demo: variogram, IDW, and leave-one-out cross-validation
Run cells top to bottom. Swap in the Lab 5 file and column names where marked.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# soil_moisture_80.csv sits beside this notebook or in data/processed/ of the course repository
DATA = Path("soil_moisture_80.csv")
if not DATA.exists():
    root = Path.cwd()
    while not (root / "data").exists() and root != root.parent:
        root = root.parent
    DATA = root / "data" / "processed" / "soil_moisture_80.csv"

points = pd.read_csv(DATA)                     # Lab 5: use your data file
x = points["x_km"].to_numpy()
y = points["y_km"].to_numpy()
z = points["moisture_pct"].to_numpy()          # Lab 5: check the column name
print(points.shape)
points.head()

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4.5))
sc = ax.scatter(x, y, c=z, cmap="YlGnBu", s=50, edgecolor="k")
fig.colorbar(sc, label="Soil moisture (%)")
ax.set_xlabel("x (km)"); ax.set_ylabel("y (km)")
ax.set_aspect("equal")
plt.show()

## 1. Distances

In [ ]:
dx, dy = 5 - 2, 7 - 3
print(np.sqrt(dx**2 + dy**2))                  # 5.0

D = np.sqrt((x[:, None] - x[None, :])**2 +
            (y[:, None] - y[None, :])**2)
# same result: from earthcourse.spatial import pairwise_distances_xy
print(D.shape, D[3, 17])

## 2. Baseline: predict the mean

In [ ]:
print(f"mean = {z.mean():.1f} %, variance = {z.var(ddof=1):.1f} %^2")
errors = []
for k in range(len(z)):
    others = np.delete(z, k)
    errors.append(others.mean() - z[k])
errors = np.array(errors)
rmse_mean = np.sqrt(np.mean(errors**2))
print(f"baseline RMSE = {rmse_mean:.2f} %")

## 3. Pairs and the variogram cloud

In [ ]:
i, j = np.triu_indices(len(z), k=1)
h = D[i, j]
g = 0.5 * (z[i] - z[j])**2
print(len(h))
plt.scatter(h, g, s=4, alpha=0.4)
plt.xlabel("Separation h (km)"); plt.ylabel("0.5 (z_i - z_j)^2")
plt.show()

## 4. Empirical variogram

In [ ]:
def empirical_variogram(h, g, bin_width=0.5, max_distance=5.0):
    edges = np.arange(0, max_distance + bin_width, bin_width)
    centers, gamma, npairs = [], [], []
    for lo, hi in zip(edges[:-1], edges[1:]):
        in_bin = (h >= lo) & (h < hi)
        if in_bin.sum() == 0:
            continue
        centers.append((lo + hi) / 2)
        gamma.append(g[in_bin].mean())
        npairs.append(in_bin.sum())
    return np.array(centers), np.array(gamma), np.array(npairs)

lags, gamma, npairs = empirical_variogram(h, g)
print(pd.DataFrame({"lag_km": lags, "gamma": gamma.round(1), "n_pairs": npairs}))

In [ ]:
fig, ax = plt.subplots()
ax.plot(lags, gamma, "o-", color="tab:red")
for lag, gam, n in zip(lags, gamma, npairs):
    ax.annotate(str(n), (lag, gam), textcoords="offset points",
                xytext=(0, 8), ha="center", fontsize=8)
ax.axhline(z.var(ddof=1), ls="--", color="k", label="sample variance")
ax.set_xlabel("Separation h (km)")
ax.set_ylabel("Semivariance (%^2)")
ax.legend()
plt.show()

## 5. Shuffle test

In [ ]:
rng = np.random.default_rng(0)
z_shuf = rng.permutation(z)
g_shuf = 0.5 * (z_shuf[i] - z_shuf[j])**2
lags_s, gamma_s, _ = empirical_variogram(h, g_shuf)
plt.plot(lags, gamma, "o-", label="real")
plt.plot(lags_s, gamma_s, "s--", label="shuffled")
plt.legend(); plt.show()

## 6. IDW at one point

In [ ]:
def idw_point(x0, y0, x, y, z, power=2.0, max_distance=None):
    d = np.sqrt((x - x0)**2 + (y - y0)**2)
    if np.any(d == 0):                      # exact hit
        return z[np.argmin(d)]
    if max_distance is not None:
        keep = d <= max_distance
        if not keep.any():
            return np.nan                   # no probe close enough
        d, z = d[keep], z[keep]
    w = 1.0 / d**power
    return np.sum(w * z) / np.sum(w)

# check against the hand example (expect 22.9)
xs = np.array([1.0, 2.0, 4.0]); ys = np.zeros(3)
zs = np.array([20.0, 30.0, 40.0])
print(idw_point(0, 0, xs, ys, zs, power=2))

## 7. IDW on a grid

In [ ]:
gx = np.linspace(0, 10, 120)
gy = np.linspace(0, 10, 120)
GX, GY = np.meshgrid(gx, gy)

def idw_grid(GX, GY, x, y, z, power=2.0):
    d = np.sqrt((GX.ravel()[:, None] - x[None, :])**2 +
                (GY.ravel()[:, None] - y[None, :])**2)
    d = np.maximum(d, 1e-12)                # avoid divide by zero
    w = 1.0 / d**power
    zhat = (w * z).sum(axis=1) / w.sum(axis=1)
    return zhat.reshape(GX.shape)

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, p in zip(axes, [1, 2, 4]):
    Zp = idw_grid(GX, GY, x, y, z, power=p)
    im = ax.pcolormesh(GX, GY, Zp, cmap="YlGnBu",
                       vmin=z.min(), vmax=z.max())
    ax.scatter(x, y, s=8, c="k")
    ax.set_title(f"p = {p}")
    ax.set_aspect("equal")
fig.colorbar(im, ax=axes, label="Soil moisture (%)")
plt.show()

## 8. Leave-one-out cross-validation

In [ ]:
def loo_idw(x, y, z, power=2.0):
    n = len(z)
    pred = np.empty(n)
    for k in range(n):
        keep = np.arange(n) != k            # everyone except k
        pred[k] = idw_point(x[k], y[k],
                            x[keep], y[keep], z[keep], power)
    return pred

def scores(pred, obs):
    err = pred - obs
    return {"MAE": np.mean(np.abs(err)),
            "RMSE": np.sqrt(np.mean(err**2)),
            "Bias": np.mean(err)}

s = scores(np.array([24, 27, 20, 25]),
           np.array([22, 30, 18, 26]))
print({k: round(float(v), 2) for k, v in s.items()})

In [ ]:
powers = [0.5, 1, 1.5, 2, 2.5, 3, 4, 6, 8]
rows = []
for p in powers:
    pred = loo_idw(x, y, z, power=p)
    rows.append({"power": p, **scores(pred, z)})
table = pd.DataFrame(rows)
print(table.round(2))
best = table.loc[table["RMSE"].idxmin(), "power"]
print("best power:", best)

In [ ]:
pred = loo_idw(x, y, z, power=best)
fig, ax = plt.subplots()
ax.scatter(z, pred, alpha=0.7)
lims = [z.min() - 1, z.max() + 1]
ax.plot(lims, lims, "r-")
ax.set_xlabel("Observed (%)"); ax.set_ylabel("LOO predicted (%)")
s = scores(pred, z)
ax.set_title(f"RMSE = {s['RMSE']:.2f} %, MAE = {s['MAE']:.2f} %")
plt.show()

## 9. Mask cells far from every probe

In [ ]:
from scipy.spatial import cKDTree
tree = cKDTree(np.column_stack([x, y]))
dist_nearest, _ = tree.query(np.column_stack([GX.ravel(), GY.ravel()]))
dist_nearest = dist_nearest.reshape(GX.shape)

Zbest = idw_grid(GX, GY, x, y, z, power=best)
max_gap = 1.0                               # km, about half the range
Z_masked = np.where(dist_nearest <= max_gap, Zbest, np.nan)
print(f"masked: {np.isnan(Z_masked).mean():.0%} of cells")
plt.pcolormesh(GX, GY, Z_masked, cmap="YlGnBu")
plt.scatter(x, y, s=8, c="k"); plt.gca().set_aspect("equal")
plt.colorbar(label="Soil moisture (%)"); plt.show()